# Shared estimators and diagnostics

Run all cells. Outputs are written to this task's `output/` folder.


In [2]:
import hashlib
import json
import math
import pickle
import re
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import least_squares, minimize
from scipy.special import expit
from scipy.stats import spearmanr
MODEL_T5 = 'SE-Hurdle-T5'
MODEL_AR = 'Unrestricted-Hurdle-AR(6,6)'
STAGES = ('0', '1-4', '5-7', '8-11')
MEMORY_STAGES = ('5-7', '8-11')
AR_ORDER = 6
MEMORY_LAGS = tuple(range(2, AR_ORDER + 1))
EARLY_INTERACTIONS = tuple((f'early_t{target}_x_lag{lag}' for target in range(2, 6) for lag in range(2, min(target, AR_ORDER) + 1)))

def read_config(root):
    return json.loads((root / 'config.json').read_text())

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, sort_keys=True) + '\n')

def sha256(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

def fingerprint(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(payload).hexdigest()

def slug(value):
    return re.sub('[^a-z0-9]+', '-', str(value).lower()).strip('-') or 'unknown'

def canonical_id(value):
    text = str(value).strip()
    if re.fullmatch('[+-]?\\d+\\.0+', text):
        text = text.split('.', 1)[0]
    if not text or text.lower() in {'nan', 'none', '<na>'}:
        raise ValueError('missing person identifier')
    return text

def stable_uniform(key, seed):
    integer = int(hashlib.sha256(f'{seed}:{key}'.encode()).hexdigest()[:13], 16)
    return integer / float(16 ** 13)

def split_for(person, seed='seh-63'):
    value = stable_uniform(person, seed)
    if value < 0.7:
        return 'train'
    if value < 0.85:
        return 'validation'
    return 'test'

def stable_seed(*parts, base=63):
    token = ':'.join(map(str, parts))
    return (base + int(hashlib.sha256(token.encode()).hexdigest()[:8], 16)) % (2 ** 32 - 1)

def stage_for_transition(age):
    age = int(age)
    if age == 0:
        return '0'
    if 1 <= age <= 4:
        return '1-4'
    if 5 <= age <= 7:
        return '5-7'
    if 8 <= age <= 11:
        return '8-11'
    raise ValueError(f'unsupported transition age {age}; expected 0..11')

def add_raw_lags(transitions, order=AR_ORDER):
    rows = transitions.sort_values(['analysis_group', 'person_domain_id', 'target_age']).copy()
    for lag in range(1, order + 1):
        rows[f'x_lag{lag}'] = 0.0
        rows[f'lag{lag}_available'] = rows['target_age'].ge(lag)
    for (_, person), indices in rows.groupby(['analysis_group', 'person_domain_id'], sort=False).groups.items():
        idx = list(indices)
        group = rows.loc[idx].sort_values('target_age')
        q_by_age = dict(zip(group.transition_age.astype(int), group.q_prev.astype(float)))
        q_by_age.update(dict(zip(group.target_age.astype(int), group.q.astype(float))))
        for index, target in zip(group.index, group.target_age.astype(int)):
            for lag in range(1, order + 1):
                age = target - lag
                if age >= 0:
                    rows.at[index, f'x_lag{lag}'] = math.log1p(max(q_by_age.get(age, 0.0), 0.0))
    for target in range(2, 6):
        for lag in range(2, min(target, order) + 1):
            name = f'early_t{target}_x_lag{lag}'
            rows[name] = rows[f'x_lag{lag}'] * rows['target_age'].eq(target).to_numpy(float)
    return rows.sort_index().reset_index(drop=True)

def add_t5_history(rows, rho):
    out = rows.copy()
    numerator = np.zeros(len(out), dtype=float)
    denominator = np.zeros(len(out), dtype=float)
    for lag in MEMORY_LAGS:
        available = out[f'lag{lag}_available'].to_numpy(bool)
        weight = float(rho ** (lag - 2))
        numerator += weight * out[f'x_lag{lag}'].to_numpy(float) * available
        denominator += weight * available
    out['history'] = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 0)
    return out
CONTINUOUS = {'history'} | {f'x_lag{k}' for k in range(1, AR_ORDER + 1)} | set(EARLY_INTERACTIONS)

def _weighted_mean(values, weights):
    return float(np.sum(values * weights) / np.sum(weights))

def fit_scaler(data, features, weights):
    means, scales = ({}, {})
    for name in features:
        if name in CONTINUOUS:
            values = data[name].to_numpy(float)
            mean = _weighted_mean(values, weights)
            variance = _weighted_mean((values - mean) ** 2, weights)
            means[name] = mean
            scales[name] = math.sqrt(variance) if variance > 1e-12 else 1.0
        else:
            means[name], scales[name] = (0.0, 1.0)
    return (means, scales)

def design_matrix(data, features, means, scales):
    columns = [np.ones(len(data), dtype=float)]
    columns.extend(((data[name].to_numpy(float) - means[name]) / scales[name] for name in features))
    return np.column_stack(columns)

def array_design(values, spec):
    n = len(next(iter(values.values())))
    columns = [np.ones(n)]
    for name in spec['feature_names']:
        columns.append((np.asarray(values[name], float) - spec['means'][name]) / spec['scales'][name])
    return np.column_stack(columns)

def fit_logistic(X, y, sample_weight, ridge, constrained_indices=None):
    weights = np.asarray(sample_weight, float)
    weight_sum = weights.sum()
    penalty = np.ones(X.shape[1])
    penalty[0] = 0.0

    def objective(beta):
        eta = np.clip(X @ beta, -35, 35)
        p = expit(eta)
        nll = -np.sum(weights * (y * np.log(p + 1e-12) + (1 - y) * np.log(1 - p + 1e-12))) / weight_sum
        value = nll + 0.5 * ridge * np.sum((penalty * beta) ** 2)
        gradient = X.T @ (weights * (p - y)) / weight_sum + ridge * penalty * beta
        return (float(value), gradient)
    bounds = [(None, None)] * X.shape[1]
    for index in constrained_indices or set():
        bounds[index] = (0.0, None)
    result = minimize(objective, np.zeros(X.shape[1]), jac=True, method='L-BFGS-B', bounds=bounds)
    if not result.success:
        raise RuntimeError(f'logistic fit failed: {result.message}')
    return result.x

def fit_linear(X, y, sample_weight, ridge, constrained_indices=None):
    weights = np.asarray(sample_weight, float)
    sqrt_w = np.sqrt(weights)
    penalty = np.eye(X.shape[1])
    penalty[0, 0] = 0.0
    X_aug = np.vstack([
        X * sqrt_w[:, None],
        math.sqrt(weights.sum() * ridge) * penalty
    ])
    y_aug = np.concatenate([y * sqrt_w, np.zeros(X.shape[1])])

    if constrained_indices:
        lower = np.full(X.shape[1], -np.inf)
        upper = np.full(X.shape[1], np.inf)

        for index in constrained_indices:
            lower[index] = 0.0

        initial = np.zeros(X.shape[1])
        initial[list(constrained_indices)] = 1e-6

        result = least_squares(
            lambda b: X_aug @ b - y_aug,
            initial,
            bounds=(lower, upper)
        )

        beta = result.x
    else:
        beta = np.linalg.lstsq(X_aug, y_aug, rcond=None)[0]

    residuals = y - X @ beta
    residuals = residuals - _weighted_mean(residuals, weights)
    sigma = max(math.sqrt(_weighted_mean(residuals ** 2, weights)), 1e-08)

    return beta, residuals, sigma

def fit_stage(data, model_name, ridge):
    if data.empty:
        raise ValueError('empty stage')
    if model_name == MODEL_T5:
        activity_features = ['x_lag1', 'prev_active', 'history']
        positive_features = ['x_lag1', 'restart', 'history']
        constrain_activity = {1 + activity_features.index('history')}
        constrain_positive = {1 + positive_features.index('history')}
    elif model_name == MODEL_AR:
        lags = [f'x_lag{k}' for k in range(1, AR_ORDER + 1)]
        early = list(EARLY_INTERACTIONS) if str(data.stage.iloc[0]) == '1-4' else []
        activity_features = lags + early + ['prev_active']
        positive_features = lags + early + ['restart']
        constrain_activity = constrain_positive = set()
    else:
        raise ValueError(model_name)
    weights = data.get('sample_weight', pd.Series(1.0, index=data.index)).to_numpy(float)
    act_means, act_scales = fit_scaler(data, activity_features, weights)
    X_act = design_matrix(data, activity_features, act_means, act_scales)
    activity_coef = fit_logistic(X_act, data.active.to_numpy(float), weights, ridge, constrain_activity)
    positive = data[data.active.eq(1)].copy()
    if len(positive) < 2:
        raise ValueError('fewer than two positive outcomes')
    pos_weights = positive.get('sample_weight', pd.Series(1.0, index=positive.index)).to_numpy(float)
    pos_means, pos_scales = fit_scaler(positive, positive_features, pos_weights)
    X_pos = design_matrix(positive, positive_features, pos_means, pos_scales)
    beta, residuals, sigma = fit_linear(X_pos, np.log(positive.q.to_numpy(float)), pos_weights, ridge, constrain_positive)
    return {'activity': {'feature_names': activity_features, 'means': act_means, 'scales': act_scales, 'coef': activity_coef, 'n': int(len(data)), 'weight_sum': float(weights.sum())}, 'positive': {'feature_names': positive_features, 'means': pos_means, 'scales': pos_scales, 'coef': beta, 'residuals': residuals, 'sigma': sigma, 'n': int(len(positive)), 'weight_sum': float(pos_weights.sum())}}

def fit_model(rows, model_name, ridge):
    model = {stage: fit_stage(rows[rows.stage.eq(stage)], model_name, ridge) for stage in STAGES}
    q0 = rows.loc[rows.transition_age.eq(0), ['person_domain_id', 'q_prev']].drop_duplicates()
    model['q0_values'] = q0.q_prev.to_numpy(float)
    return model

def original_coefficients(spec):
    output = {'intercept': float(spec['coef'][0])}
    for index, name in enumerate(spec['feature_names'], start=1):
        output[name] = float(spec['coef'][index] / spec['scales'][name])
        output['intercept'] -= float(spec['coef'][index] * spec['means'][name] / spec['scales'][name])
    return output

def parameter_table(model, analysis_group, model_name, replicate=None):
    records = []
    for stage in STAGES:
        for equation in ('activity', 'positive'):
            spec = model[stage][equation]
            records.append({'analysis_group': analysis_group, 'model': model_name, 'stage': stage, 'equation': equation, 'n_rows': spec['n'], 'weight_sum': spec['weight_sum'], 'sigma': spec.get('sigma', np.nan), 'replicate': replicate, **original_coefficients(spec)})
    return pd.DataFrame(records)

def score_rows(model, rows):
    pieces = []
    for stage in STAGES:
        data = rows[rows.stage.eq(stage)].copy()
        if data.empty:
            continue
        act = model[stage]['activity']
        p = np.clip(expit(design_matrix(data, act['feature_names'], act['means'], act['scales']) @ act['coef']), 1e-09, 1 - 1e-09)
        y = data.active.to_numpy(float)
        extensive = -(y * np.log(p) + (1 - y) * np.log(1 - p))
        positive_nll = np.zeros(len(data))
        mask = data.active.eq(1).to_numpy()
        if mask.any():
            pos_data = data.loc[mask]
            pos = model[stage]['positive']
            mean = design_matrix(pos_data, pos['feature_names'], pos['means'], pos['scales']) @ pos['coef']
            z = (np.log(pos_data.q.to_numpy(float)) - mean) / pos['sigma']
            positive_nll[mask] = 0.5 * z ** 2 + np.log(pos['sigma']) + 0.5 * np.log(2 * np.pi)
        part = data[['analysis_group', 'person_domain_id', 'source_person_id', 'stage', 'split']].copy()
        part['extensive_nll'] = extensive
        part['positive_nll'] = positive_nll
        part['total_nll'] = extensive + positive_nll
        pieces.append(part)
    return pd.concat(pieces, ignore_index=True)

def mean_nll(model, rows):
    scored = score_rows(model, rows)
    return float(scored.total_nll.mean())

def cluster_bootstrap_gap(person_scores, seed, n_boot=5000):
    wide = person_scores.pivot(index='person_domain_id', columns='model', values='mean_nll').dropna()
    delta = wide[MODEL_AR].to_numpy(float) - wide[MODEL_T5].to_numpy(float)
    if not len(delta):
        return {'delta_ar_minus_t5': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'people': 0}
    rng = np.random.default_rng(seed)
    draws = np.empty(n_boot)
    for b in range(n_boot):
        draws[b] = rng.choice(delta, size=len(delta), replace=True).mean()
    return {'delta_ar_minus_t5': float(delta.mean()), 'ci_low': float(np.quantile(draws, 0.025)), 'ci_high': float(np.quantile(draws, 0.975)), 'people': int(len(delta))}

def bootstrap_weights(rows, rng):
    people = rows.person_domain_id.drop_duplicates().to_numpy()
    sampled = rng.choice(people, size=len(people), replace=True)
    counts = pd.Series(sampled).value_counts()
    out = rows[rows.person_domain_id.isin(counts.index)].copy()
    out['sample_weight'] = out.person_domain_id.map(counts).astype(float)
    return out

def _shape_prediction(kind, lag, amplitude, shape):
    x = lag - 2.0
    if kind == 'geometric':
        return amplitude * shape ** x
    if kind == 'linear':
        return amplitude * (1.0 - shape * x)
    if kind == 'power':
        return amplitude * (lag - 1.0) ** (-shape)
    if kind == 'hyperbolic':
        return amplitude / (1.0 + shape * x)
    if kind == 'flat':
        return np.full_like(lag, amplitude, dtype=float)
    raise ValueError(kind)

def fit_shared_shape(points, kind='geometric', se_col=None):
    points = points[points.stage.isin(MEMORY_STAGES) & points.lag.isin(MEMORY_LAGS)].dropna(subset=['weight']).copy()
    stages = [s for s in MEMORY_STAGES if s in set(points.stage)]
    if len(stages) < 1 or len(points) < 4:
        return {'shape': np.nan, 'rho': np.nan, 'half_life': np.nan, 'rmse': np.nan, 'r2': np.nan, 'aicc': np.nan, 'n_points': len(points), 'n_parameters': np.nan}
    lags = points.lag.to_numpy(float)
    observed = points.weight.to_numpy(float)
    stage_index = np.array([stages.index(s) for s in points.stage])
    initial_amp = [float(points.loc[points.stage.eq(s), 'weight'].iloc[0]) for s in stages]
    if kind == 'flat':
        x0 = np.asarray(initial_amp)
        lower = np.full(len(stages), -np.inf)
        upper = np.full(len(stages), np.inf)
    else:
        initial_shape = 0.65 if kind == 'geometric' else 0.15 if kind == 'linear' else 1.0
        x0 = np.asarray(initial_amp + [initial_shape])
        lower = np.r_[np.full(len(stages), -np.inf), 0.0]
        upper_shape = 0.249999 if kind == 'linear' else 0.999999 if kind == 'geometric' else 20.0
        upper = np.r_[np.full(len(stages), np.inf), upper_shape]
    if se_col and se_col in points:
        scale = points[se_col].to_numpy(float)
        finite = scale[np.isfinite(scale) & (scale > 1e-10)]
        floor = np.median(finite) * 0.25 if len(finite) else 1.0
        scale = np.where(np.isfinite(scale) & (scale > floor), scale, floor)
    else:
        stage_scale = {s: max(float(np.sqrt(np.mean(points.loc[points.stage.eq(s), 'weight'] ** 2))), 1e-06) for s in stages}
        scale = np.array([stage_scale[s] for s in points.stage])

    def residual(theta):
        amps = theta[:len(stages)]
        shape = 0.0 if kind == 'flat' else theta[-1]
        prediction = np.array([_shape_prediction(kind, np.array([lag]), amps[idx], shape)[0] for lag, idx in zip(lags, stage_index)])
        return (prediction - observed) / scale
    fit = least_squares(residual, x0, bounds=(lower, upper), max_nfev=10000)
    amps = fit.x[:len(stages)]
    shape = 0.0 if kind == 'flat' else float(fit.x[-1])
    prediction = np.array([_shape_prediction(kind, np.array([lag]), amps[idx], shape)[0] for lag, idx in zip(lags, stage_index)])
    residual_raw = observed - prediction
    rss = max(float(np.sum(residual_raw ** 2)), 1e-20)
    tss = float(np.sum((observed - observed.mean()) ** 2))
    n = len(observed)
    k = len(fit.x)
    aic = n * math.log(rss / n) + 2 * k
    aicc = aic + (2 * k * (k + 1) / (n - k - 1) if n > k + 1 else np.inf)
    rho = shape if kind == 'geometric' else np.nan
    half_life = math.log(0.5) / math.log(rho) if 0 < rho < 1 else np.nan
    return {'shape': shape, 'rho': rho, 'half_life': half_life, 'rmse': math.sqrt(rss / n), 'r2': 1 - rss / tss if tss > 0 else np.nan, 'aicc': float(aicc), 'n_points': n, 'n_parameters': k, **{f'amplitude_{slug(stage)}': float(amps[i]) for i, stage in enumerate(stages)}}

def coefficient_points(parameters, analysis_group, equation):
    rows = parameters[parameters.analysis_group.eq(analysis_group) & parameters.model.eq(MODEL_AR) & parameters.equation.eq(equation)]
    records = []
    for row in rows.itertuples(index=False):
        for lag in MEMORY_LAGS:
            records.append({'analysis_group': analysis_group, 'equation': equation, 'stage': row.stage, 'lag': lag, 'weight': getattr(row, f'x_lag{lag}'), 'replicate': getattr(row, 'replicate', None)})
    return pd.DataFrame(records)

def exact_spearman_permutation(x, y):
    from itertools import permutations
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if len(np.unique(x)) < 2 or len(np.unique(y)) < 2:
        return (np.nan, np.nan)
    observed = float(spearmanr(x, y).statistic)
    if len(x) <= 8:
        values = [abs(float(spearmanr(x, perm).statistic)) for perm in permutations(y)]
        p = sum((v >= abs(observed) - 1e-12 for v in values)) / len(values)
    else:
        p = float(spearmanr(x, y).pvalue)
    return (observed, float(p))

def t5_nesting_max_error(rows, rho):
    target_history = add_t5_history(rows, rho)['history'].to_numpy(float)
    reconstructed = np.zeros(len(rows), dtype=float)
    for target in range(2, 6):
        denominator = sum((rho ** (lag - 2) for lag in range(2, target + 1)))
        for lag in range(2, target + 1):
            reconstructed += rho ** (lag - 2) / denominator * rows[f'early_t{target}_x_lag{lag}'].to_numpy(float)
    full_denominator = sum((rho ** (lag - 2) for lag in MEMORY_LAGS))
    full = rows.target_age.ge(6).to_numpy(float)
    for lag in MEMORY_LAGS:
        reconstructed += full * (rho ** (lag - 2) / full_denominator) * rows[f'x_lag{lag}'].to_numpy(float)
    return float(np.max(np.abs(target_history - reconstructed)))

def save_bundle(path, bundle):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('wb') as stream:
        pickle.dump(bundle, stream)

def load_bundle(path):
    with path.open('rb') as stream:
        return pickle.load(stream)

def t5_history_from_simulation(q, target, rho):
    values, weights = ([], [])
    for lag in MEMORY_LAGS:
        age = target - lag
        if age >= 0:
            values.append(np.log1p(q[:, age]))
            weights.append(rho ** (lag - 2))
    if not values:
        return np.zeros(len(q))
    weights = np.asarray(weights, float)
    return np.column_stack(values) @ weights / weights.sum()

def simulate(model, model_name, rho, n_sim, horizon, seed, observed_lengths=None):
    rng = np.random.default_rng(seed)
    q = np.zeros((n_sim, horizon + 1), float)
    q[:, 0] = rng.choice(model['q0_values'], size=n_sim, replace=True)
    for target in range(1, horizon + 1):
        previous = q[:, target - 1]
        lag_values = {}
        for lag in range(1, AR_ORDER + 1):
            age = target - lag
            lag_values[f'x_lag{lag}'] = np.log1p(q[:, age]) if age >= 0 else np.zeros(n_sim)
        early_values = {}
        for early_target in range(2, 6):
            for lag in range(2, early_target + 1):
                early_values[f'early_t{early_target}_x_lag{lag}'] = lag_values[f'x_lag{lag}'] if target == early_target else np.zeros(n_sim)
        if model_name == MODEL_T5:
            values_act = {'x_lag1': lag_values['x_lag1'], 'prev_active': (previous > 0).astype(float), 'history': t5_history_from_simulation(q, target, rho)}
            values_pos = {'x_lag1': lag_values['x_lag1'], 'restart': (previous <= 0).astype(float), 'history': values_act['history']}
        else:
            values_act = {**lag_values, **early_values, 'prev_active': (previous > 0).astype(float)}
            values_pos = {**lag_values, **early_values, 'restart': (previous <= 0).astype(float)}
        fitted = model[stage_for_transition(target - 1)]
        activity = fitted['activity']
        p = expit(np.clip(array_design(values_act, activity) @ activity['coef'], -35, 35))
        active = rng.random(n_sim) < p
        if active.any():
            positive = fitted['positive']
            active_values = {name: np.asarray(value)[active] for name, value in values_pos.items()}
            mean = array_design(active_values, positive) @ positive['coef']
            q[active, target] = np.exp(np.clip(mean + rng.normal(0, positive['sigma'], active.sum()), -30, 30))
    if observed_lengths is None:
        lengths = np.full(n_sim, horizon + 1, int)
    else:
        lengths = rng.choice(np.asarray(observed_lengths, int), size=n_sim, replace=True)
        for i, length in enumerate(lengths):
            q[i, int(length):] = np.nan
    return (q, lengths)

def notebook_code_sha256(path):
    notebook = json.loads(path.read_text())

    code = "\n".join(
        "".join(cell["source"])
        for cell in notebook["cells"]
        if cell["cell_type"] == "code"
    )

    return hashlib.sha256(code.encode()).hexdigest()